In [1]:
# ===============================================
# 1. Import Libraries
# ===============================================
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
# ===============================================
# 2. Load Dataset
# ===============================================
df = pd.read_csv("../dataset/kc_house_data.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (21613, 21)


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [3]:
# ===============================================
# 3. Check Columns
# ===============================================
print("Columns in Dataset:")
print(df.columns)

Columns in Dataset:
Index(['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqft_living15', 'sqft_lot15'],
      dtype='object')


In [4]:
# ===============================================
# 4. Check Duplicate Rows
# ===============================================
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")

print("Dataset shape after removing duplicates:", df.shape)

Duplicate rows: 0
Dataset shape after removing duplicates: (21613, 21)


In [5]:
# ===============================================
# 5. Dataset Information
# ===============================================
print("Dataset Info:")
df.info()

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  object 
 2   price          21613 non-null  float64
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float

In [6]:
# ===============================================
# 6. Convert Date Column
# ===============================================
df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S')

In [7]:
# ===============================================
# 7. Extract Date Features
# ===============================================
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.month

In [8]:
# ===============================================
# 8. Remove unnecessary columns
# ===============================================
df.drop(columns=['id','date'], inplace=True)

In [9]:
# ===============================================
# 9. Handle Invalid Values
# ===============================================

original_rows = df.shape[0]

# Bedrooms must be > 0
df = df[df['bedrooms'] > 0]

# Bathrooms must be > 0
df = df[df['bathrooms'] >= 0.5]

# Living area must be positive
df = df[df['sqft_living'] > 0]

# Lot size must be positive
df = df[df['sqft_lot'] > 0]

# Remove extreme bedroom outliers
df = df[df['bedrooms'] <= 11]

removed_rows = original_rows - df.shape[0]
print("Rows removed due to invalid values:", removed_rows)
print("Dataset shape after cleaning:", df.shape)

Rows removed due to invalid values: 17
Dataset shape after cleaning: (21596, 21)


In [10]:
# ===============================================
# 10. Check Missing Values
# ===============================================
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
sale_year        0
sale_month       0
dtype: int64


In [11]:
# ===============================================
# 11. Feature Engineering
# ===============================================

# Total house area
df['total_sqft'] = df['sqft_above'] + df['sqft_basement']

# Basement ratio
df["basement_ratio"] = np.where(
    df["total_sqft"] == 0,
    0,
    df["sqft_basement"] / df["total_sqft"]
)

# House age
df['house_age'] = df['sale_year'] - df['yr_built']
df['house_age'] = df['house_age'].clip(lower=0)

# Renovated indicator
df['renovated'] = df['yr_renovated'].apply(lambda x: 0 if x == 0 else 1)

# Years since renovation
df["years_since_renovation"] = np.where(
    df["yr_renovated"] == 0,
    0,
    df["sale_year"] - df["yr_renovated"]
)
df["years_since_renovation"] = df["years_since_renovation"].clip(lower=0)

# Log transformation of target
df["price_log"] = np.log1p(df["price"])

In [12]:
# ===============================================
# 12. Statistical Summary
# ===============================================
print("Statistical Summary:")
df.describe()

Statistical Summary:


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,sale_year,sale_month,total_sqft,basement_ratio,house_age,renovated,years_since_renovation,price_log
count,2.159600e+04,21596.000000,21596.000000,21596.000000,2.159600e+04,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000,21596.000000
mean,5.401983e+05,3.371828,2.115843,2080.343165,1.509983e+04,1.494119,0.007548,0.234303,3.409752,7.657946,1788.631506,291.711660,1971.000787,84.468698,98077.950685,47.560087,-122.213977,1986.650722,12758.656649,2014.322976,6.573995,2080.343165,0.124584,43.322745,0.042323,0.780839,13.048130
std,3.671416e+05,0.904114,0.768998,918.122038,4.141355e+04,0.539685,0.086551,0.766406,0.650471,1.173218,827.763251,442.673703,29.375460,401.830330,53.514040,0.138552,0.140725,685.231768,27275.018316,0.467625,3.115131,918.122038,0.170996,29.376694,0.201329,4.897135,0.526424
min,7.800000e+04,1.000000,0.500000,370.000000,5.200000e+02,1.000000,0.000000,0.000000,1.000000,3.000000,370.000000,0.000000,1900.000000,0.000000,98001.000000,47.155900,-122.519000,399.000000,651.000000,2014.000000,1.000000,370.000000,0.000000,0.000000,0.000000,0.000000,11.264477
25%,3.220000e+05,3.000000,1.750000,1430.000000,5.040000e+03,1.000000,0.000000,0.000000,3.000000,7.000000,1190.000000,0.000000,1951.000000,0.000000,98033.000000,47.471100,-122.328000,1490.000000,5100.000000,2014.000000,4.000000,1430.000000,0.000000,18.000000,0.000000,0.000000,12.682310
50%,4.500000e+05,3.000000,2.250000,1910.000000,7.619000e+03,1.500000,0.000000,0.000000,3.000000,7.000000,1560.000000,0.000000,1975.000000,0.000000,98065.000000,47.571800,-122.231000,1840.000000,7620.000000,2014.000000,6.000000,1910.000000,0.000000,40.000000,0.000000,0.000000,13.017005
75%,6.450000e+05,4.000000,2.500000,2550.000000,1.068550e+04,2.000000,0.000000,0.000000,4.000000,8.000000,2210.000000,560.000000,1997.000000,0.000000,98118.000000,47.678000,-122.125000,2360.000000,10083.000000,2015.000000,9.000000,2550.000000,0.274312,63.000000,0.000000,0.000000,13.377007
max,7.700000e+06,11.000000,8.000000,13540.000000,1.651359e+06,3.500000,1.000000,4.000000,5.000000,13.000000,9410.000000,4820.000000,2015.000000,2015.000000,98199.000000,47.777600,-121.315000,6210.000000,871200.000000,2015.000000,12.000000,13540.000000,0.666667,115.000000,1.000000,80.000000,15.856731


In [13]:
# ===============================================
# 13. Price Distribution Information
# ===============================================
print("Price Statistics:")
print("Minimum Price:", df['price'].min())
print("Maximum Price:", df['price'].max())
print("Mean Price:", df['price'].mean())
print("Median Price:", df['price'].median())

print("\nTarget Skewness:")
print("Price skewness:", df["price"].skew())
print("Log Price skewness:", df["price_log"].skew())

Price Statistics:
Minimum Price: 78000.0
Maximum Price: 7700000.0
Mean Price: 540198.2986664197
Median Price: 450000.0

Target Skewness:
Price skewness: 4.025670312013785
Log Price skewness: 0.43018955350575233


In [14]:
# ===============================================
# 14. Outlier Detection (Price using IQR)
# ===============================================
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Outlier Detection (Price)")
print("Lower Bound:", lower)
print("Upper Bound:", upper)

outliers = df[(df['price'] < lower) | (df['price'] > upper)]

print("Number of Outliers:", outliers.shape[0])
print("Outlier Percentage:", round((outliers.shape[0] / df.shape[0]) * 100, 2), "%")

Outlier Detection (Price)
Lower Bound: -162500.0
Upper Bound: 1129500.0
Number of Outliers: 1145
Outlier Percentage: 5.3 %


In [15]:
# ===============================================
# 15. Identify Column Types (Useful for EDA)
# ===============================================
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

print("Numeric Columns:", len(numeric_cols))
print("Categorical Columns:", len(categorical_cols))

Numeric Columns: 27
Categorical Columns: 0


In [16]:
# ===============================================
# 16. Feature Cardinality
# ===============================================
print("\nUnique values in each column:")
print(df.nunique().sort_values())


Unique values in each column:
waterfront                   2
sale_year                    2
renovated                    2
view                         5
condition                    5
floors                       6
grade                       11
bedrooms                    11
sale_month                  12
bathrooms                   29
yr_renovated                70
zipcode                     70
years_since_renovation      70
yr_built                   116
house_age                  116
sqft_basement              306
long                       751
sqft_living15              777
sqft_above                 942
sqft_living               1034
total_sqft                1034
price_log                 4024
price                     4024
basement_ratio            4886
lat                       5033
sqft_lot15                8682
sqft_lot                  9776
dtype: int64


In [17]:
# Validate house age
df = df[df['house_age'] <= 120]

# Validate renovation year
df = df[df['yr_renovated'] <= df['sale_year']]

In [18]:
# ===============================================
# 17. Final Dataset Validation
# ===============================================
print("\nFinal Dataset Checks")

print("Duplicate rows:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum())
print("Negative prices:", (df['price'] < 0).sum())

# Remove duplicates again if created
df = df.drop_duplicates()

print("Final Dataset Shape:", df.shape)

print("Infinite values:", np.isinf(df.select_dtypes(include=np.number)).sum().sum())


Final Dataset Checks
Duplicate rows: 2
Missing values: 0
Negative prices: 0
Final Dataset Shape: (21588, 27)
Infinite values: 0


In [19]:
# ===============================================
# 18. Save Clean Dataset
# ===============================================
df.to_csv("../dataset/cleaned_dataset.csv", index=False)
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
